<p><font size="6" color='grey'> <b>
Machine Learning
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Neural Network - MLPRegressor - Diamonds
</b></font> </br></p>

---


# 0  | Install & Import
***

In [ ]:
# Install

In [ ]:
# Daten
from pandas import read_csv, DataFrame, concat

# Preprocessing
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

# Feature Engineering / Auswahl
from sklearn.decomposition import PCA
from sklearn.feature_selection import f_regression

# Modellvorbereitung
from sklearn.model_selection import train_test_split

# Modell (Regression)
from sklearn.neural_network import MLPRegressor

# Evaluation
from sklearn.metrics import r2_score, mean_absolute_error

# Visualisierung
import plotly.express as px
import plotly.subplots as sp

# Modell-Analyse / Diagnose
from yellowbrick.regressor import residuals_plot, prediction_error
from yellowbrick.model_selection import feature_importances

In [ ]:
# Warnung ausstellen
import warnings
warnings.filterwarnings("ignore")

# 1 | Understand
---


<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Aufgabe verstehen</br>
✅ Daten sammeln</br>
✅ Statistische Analyse (Min, Max, Mean, Korrelation, ...)</br>
✅ Datenvisualisierung (Streudiagramm, Box-Plot, ...)</br>
✅ Prepare Schritte festlegen</br>

<p><font color='black' size="5">
Anwendungsfall
</font></p>

---

Dieser klassische Datensatz enthält die Preise und andere Attribute von fast 54.000 Diamanten.



[DataSet](https://www.openml.org/search?type=data&status=active&id=42225)

[Info](https://www.kaggle.com/datasets/shivam2503/diamonds)


In [ ]:
df = read_csv(
    "https://raw.githubusercontent.com/ralf-42/ML_Intro/main/02_daten/05_tabellen/diamonds.csv",
    usecols=[
        "carat",
        "cut",
        "color",
        "clarity",
        "depth",
        "table",
        "price",
    ],
)

In [ ]:
data = df.copy()
target = data.pop("price")

<p><font color='black' size="5">
EDA (Exploratory Data Analysis) mit Pandas
</font></p>

In [ ]:
data.info()

In [ ]:
data.describe().T

In [ ]:
data.groupby("cut").count()

In [ ]:
data.groupby("color").count()

<p><font color='black' size="5">
EDA (Exploratory Data Analysis) mit Plotly
</font></p>

In [ ]:
title_ = "Depth"
b1 = px.box(data["depth"], title=title_, width=600, height=600)

title_ = "Carat"
b2 = px.box(data["carat"], title=title_, width=600, height=600)

title_ = "Table"
b3 = px.box(data["table"], title=title_, width=600, height=600)

fig = sp.make_subplots(rows=1, cols=3, subplot_titles=("Depth", "Carat", "Table"))

for trace in b1.data:
    fig.add_trace(trace, row=1, col=1)

for trace in b2.data:
    fig.add_trace(trace, row=1, col=2)

for trace in b3.data:
    fig.add_trace(trace, row=1, col=3)

# Layout anpassen
fig.update_layout(width=1000, height=500, title_text="Box-Plots")

# Plot anzeigen
fig.show()

# 2 |  Prepare

---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Datentyp ermitteln/ändern</br>
✅ Train-Test-Split durchführen</br>
✅ Nicht benötigte Features löschen</br>
✅ Missing Values behandeln</br>
✅ Ausreißer behandeln</br>
✅ Kategorischer Features Kodieren</br>
✅ Numerischer Features skalieren</br>
✅ Feature-Engineering (neue Features schaffen)</br>
✅ Dimensionalität reduzieren</br>
✅ Resampling (Over-/Undersampling)</br>
✅ Pipeline erstellen/konfigurieren</br>

<p><font color='black' size="5">
Datentyp ermitteln
</font></p>

In [ ]:
all_col = data.columns
num_col = data.select_dtypes(include="number").columns
cat_col = data.select_dtypes(exclude="number").columns

<p><font color='black' size="5">
Train-Test-Set
</font></p>


In [ ]:
data_train, data_test, target_train, target_test = train_test_split(
    data, target, test_size=0.2, random_state=42)

<p><font color='black' size="5">
Kodierung
</font></p>

In [ ]:
cat_seq = [
    ["Fair", "Good", "Very Good", "Premium", "Ideal"],
    ["J", "I", "H", "G", "F", "E", "D"],
    ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"],]

encoder = OrdinalEncoder(categories=cat_seq)
encoder.fit(data_train[cat_col])
data_train[cat_col] = encoder.transform(data_train[cat_col])
data_test[cat_col] = encoder.transform(data_test[cat_col])

<font color='black' size="5">
Skalierung
</font>

In [ ]:
scaler = StandardScaler()
scaler.fit(data_train[all_col])
data_train[all_col] = scaler.transform(data_train[all_col])
data_test[all_col] = scaler.transform(data_test[all_col])

# 3 | Modeling
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellauswahl treffen</br>
✅ Pipeline erweitern/konfigurieren</br>
✅ Training durchführen</br>
✅ Hyperparameter Tuning</br>
✅ Cross-Valdiation</br>
✅ Bootstrapping</br>
✅ Regularization</br>

<p><font color='black' size="5">🧠 Algorithmus: MLP – Mehrschichtiges Perzeptron (Regression)</font></p>

Ein MLPRegressor ist ein künstliches neuronales Netz für Regressionsprobleme. Im Gegensatz zur Klassifikation sagt das Modell einen kontinuierlichen Zahlenwert vorher (z.B. einen Preis oder Messwert). Die Netzwerkstruktur ist identisch zum MLPClassifier.

**Analogie:** Wie in einer Fabrik werden Informationen schrittweise verarbeitet – am Ende steht kein Label, sondern ein Messwert.

**Wichtige Parameter:**

| Parameter | Bedeutung |
|-----------|----------|
| `hidden_layer_sizes` | Anzahl und Größe der versteckten Schichten |
| `activation` | Aktivierungsfunktion (z.B. `relu`) |
| `max_iter` | Maximale Trainings-Epochen |
| `random_state` | Reproduzierbarkeit |

**In der Praxis relevant wenn:**
- Kontinuierliche Zahlenwerte (z.B. Preise, Temperaturen) vorhergesagt werden sollen und nichtlineare Zusammenhänge vermutet werden
- Ein neuronales Netz ohne aufwändige Keras-Konfiguration für Regressionsprobleme gebraucht wird
- Der Datensatz mittelgroß ist und ausschließlich numerische Features (nach Skalierung) enthält

**Nicht geeignet wenn:**
- Sehr viele Features bei wenigen Datenpunkten vorliegen → dann lineare Regression mit Regularisierung
- Das Modell vollständig interpretierbar sein muss → Regressionskoeffizienten (z.B. bei Ridge) sind nachvollziehbarer

**Typischer Fehler:**
`max_iter` zu niedrig setzen — der MLPRegressor konvergiert oft erst nach vielen Epochen. Wenn der Loss noch nicht stagniert, `max_iter` erhöhen oder `learning_rate="adaptive"` nutzen.

In [ ]:
#@markdown   <p><font size="4" color='green'>  MLP – Netzwerkstruktur (Regression)</font> </br></p>

import base64
from IPython.display import Image, display

diagram = """
graph LR
    subgraph Input_Layer [Eingabeschicht]
        IN[Eingangs-Features]
    end

    subgraph Hidden_Layers [Verborgene Schichten]
        direction TB
        H1[Layer 1: 88 Neuronen]
        H2[Layer 2: 44 Neuronen]
        H3[Layer 3: 22 Neuronen]
    end

    subgraph Output_Layer [Ausgabeschicht]
        OUT[Numerischer Wert / Linear]
    end

    %% Datenfluss
    IN --> H1
    H1 -->|ReLU| H2
    H2 -->|ReLU| H3
    H3 --> OUT

    %% Parameter-Visualisierung
    classDef config fill:#fcf,stroke:#333,stroke-width:1px;

    PARAM[<b>Konfiguration</b><br/>Solver: adam<br/>Learning Rate: adaptive<br/>Alpha: 0.15<br/>Max Iter: 25]
    class PARAM config
"""

encoded = base64.urlsafe_b64encode(diagram.strip().encode()).decode()
display(Image(url=f"https://mermaid.ink/img/{encoded}", width=950))

 <p><font color='black' size="5">
Modellauswahl & Training
</font></p>

In [ ]:
model = MLPRegressor(
    max_iter=25,
    hidden_layer_sizes=(88, 44, 22),
    activation="relu",
    solver="adam",
    learning_rate="adaptive",
    alpha=0.15,
    verbose=1,
    random_state=42,
)

In [ ]:
model.fit(data_train, target_train)

<p><font color='black' size="5">
Loss-Entwickung
</font></p>

In [ ]:
title_ = "Loss-Entwicklung"
px.line(
    y=model.loss_curve_,
    title=title_,
    labels={"x": "Epochen", "y": "Loss-Wert"},
    width=800,
    height=400,
)

# 4 | Evaluate
---


<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Prognose (Train, Test) erstellen</br>
✅ Modellgüte prüfen</br>
✅ Residuenanalyse erstellen</br>
✅ Feature Importance/Selektion prüfen</br>
✅ Robustheitstest erstellen</br>
✅ Modellinterpretation erstellen</br>
✅ Sensitivitätsanalyse erstellen</br>
✅ Kommunikation (Key Takeaways)</br>


<p><font color='black' size="5">
Prediction
</font></p>


In [ ]:
target_train_pred = model.predict(data_train)
target_test_pred = model.predict(data_test)

<p><font color='black' size="5">
Bestimmtheitsmass
</font></p>

In [ ]:
r2 = r2_score(target_train, target_train_pred)
print(f"Modell: {model} -- Train --- Bestimmtheitsmass: {r2:5.2f}")

In [ ]:
r2 = r2_score(target_test, target_test_pred)
print(f"Modell: {model} -- Test --- Bestimmtheitsmass: {r2:5.2f}")

<p><font color='black' size="5">
Mean Absolut Error
</font></p>

In [ ]:
mae = mean_absolute_error(target_test, target_test_pred)
print(f"Modell: {model} -- Test -- Mean Absolute Error: {mae:5.2f}")

<p><font color='black' size="5">
Aufbau Analysewürfel
</font></p>

In [ ]:
# Übernahme der Testdaten
cube = data_test.copy()
cube.reset_index(inplace=True)

# Übernahem Target real & predict
cube["real"] = DataFrame(target_test.values, columns=["real"])
cube["predict"] = DataFrame(target_test_pred, columns=["predict"])

<p><font color='black' size="5">
Visualisierung real vs predict
</font></p>

In [ ]:
# Boxplot
title_ = "Boxplot real vs predict"
px.box(cube[["real", "predict"]], title=title_, width=600, height=600)

In [ ]:
# Histogramm
title_ = "Histogramm Prices real vs predict"
fig = px.histogram(cube, x=["real", "predict"], nbins=10, text_auto=".2s", title=title_)
fig.update_layout(barmode="group", bargap=0.2, width=800, height=600)
fig.show()

<p><font color='black' size="5">
Fehlerhafte Vorhersagen
</font></p>

In [ ]:
cube["abs_Abw%"] = abs((cube["real"] - cube["predict"]) / cube["real"] * 100)
%precision 3
cube.head(10).style.format("{:,.1f}")

In [ ]:
cube.describe().T

In [ ]:
# Histogramm
title_ = "Histogramm absolute Abweichung"
fig = px.histogram(cube, x=["abs_Abw%"], nbins=10, text_auto=".2s", title=title_)
fig.update_layout(barmode="group", bargap=0.2, width=800, height=600)
fig.show()

<p><font color='black' size="5">
Residual Plot
</font></p>

In [ ]:
_ = residuals_plot(model, data_train, target_train, data_test, target_test)

<p><font color='black' size="5">
Feature Importance
</font></p>

In [ ]:
fscores, pvalues = f_regression(data_test, target_test)
for i in range(len(fscores)):
    print(
        f"Feature {i+1}: {data.columns[i]:10s} score = {fscores[i]:>12,.2f}, p-value = {pvalues[i]:.3f}"
    )

In [ ]:
px.bar(x=fscores, y=data.columns, width=600, height=600).update_yaxes(
    categoryorder="total ascending"
)

# 5 | Deploy
---

<p><font color='black' size="5">📋 Checkliste</font></p>

✅ Modellexport und -speicherung</br>
✅ Abhängigkeiten und Umgebung</br>
✅ Sicherheit und Datenschutz</br>
✅ In die Produktion integrieren</br>
✅ Tests und Validierung</br>
✅ Dokumentation & Wartung</br>